# 01 — Download and audit
Baixa revisões rastreáveis e produz manifests. Defina `EXCLUDE_GLINT=True` se a licença for incompatível com o uso pretendido.

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError("Adicione HF_TOKEN em Colab > Secrets e habilite o acesso ao notebook")
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ['HF_TOKEN'] = HF_TOKEN
print('Hugging Face autenticado; o token não será exibido nem salvo nos manifests.')

In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT_ROOT = Path(os.environ.get('FABLE_PROJECT_ROOT', '/content/logos-3')).resolve()
os.chdir(PROJECT_ROOT)
EXCLUDE_GLINT = False
command = [sys.executable, 'scripts/download_datasets.py', '--config', 'configs/data.yaml']
if EXCLUDE_GLINT:
    command.append('--exclude-glint')
subprocess.run(command, check=True)
subprocess.run([sys.executable, 'scripts/audit_datasets.py', '--config', 'configs/data.yaml'], check=True)

In [ ]:
import json
for manifest in sorted(Path('data/manifests').glob('*.json')):
    data = json.loads(manifest.read_text(encoding='utf-8'))
    print(manifest, data.get('resolved_revision', data.get('splits')), data.get('license_card', data.get('license')))